<h2>Import Libraries</h2>

In [60]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder


<h2>Load Dataset</h2>

In [61]:
df = pd.read_csv('../data/raw_data/loan_data_raw.csv')

<h2>Data Cleaning</h2>

In [ ]:
# Rename key columns, then standardize all column names to lowercase

rename_columns={
    'LoanAmount': 'loan_amount',
    'CoapplicantIncome': 'coapplicant_income',
    'ApplicantIncome': 'applicant_income'
}

df.rename(columns=rename_columns,inplace=True)

def standardize_columns(df):
    df.columns = (
        df.columns.str.strip()              
                 .str.lower()               
                 
    )
    return df

df = standardize_columns(df)
print(df.columns)

df.info()

In [ ]:
# Convert 3+ to 3 and cast dependents to a clean nullable integer

(df["dependents"] == "3+").sum()

df["dependents"] = df["dependents"].replace("3+", "3")

df["dependents"] = pd.to_numeric(df["dependents"])

df["dependents"] = df["dependents"].astype("Int64")

df["dependents"].unique()

df["dependents"].dtype

In [64]:
# Convert credit_history to categorical dtype
df['credit_history'] = df['credit_history'].astype('category')

In [65]:
# Impute missing values in categorical columns using the most frequent value

cat_cols = ['gender', 'married', 'dependents', 'self_employed', 'credit_history']
for col in cat_cols:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df[[col]] = cat_imputer.fit_transform(df[[col]])

In [66]:
# Impute missing values in numeric columns using the median

num_cols = ['loan_amount', 'loan_amount_term']
num_imputer = SimpleImputer(strategy='median')
df[num_cols] = num_imputer.fit_transform(df[num_cols])

In [ ]:
print("Total missing values remaining:", df.isnull().sum())

<h2>Data Preprocessing</h2>

In [68]:
# Combine base categorical columns with education and property_area
cat_features_all = cat_cols + ['education', 'property_area']

# Combine base numeric columns with applicant and coapplicant income 
num_cols_all = num_cols + ['applicant_income', 'coapplicant_income']

In [ ]:
# Fit and transform categorical columns into a one-hot encoded array

encoder = OneHotEncoder(
    sparse_output=False,
    drop='first',
    handle_unknown='ignore'
)
encoded_dat=encoder.fit_transform(df[cat_cols])
encoded_dat

In [70]:
# Get encoded column names and label-encode the target variable

encoded_cols = encoder.get_feature_names_out(cat_cols)
target_encoder = LabelEncoder()
encoded_dat_loan_stat = target_encoder.fit_transform(
    df["loan_status"]
)

In [71]:
df['loan_status'] = encoded_dat_loan_stat

In [72]:
# Convert encoded array into a labeled integer DataFrame
cat_encoded_features  = pd.DataFrame(encoded_dat, columns=encoded_cols).astype(int)

In [73]:
# Standardize column names to lowercase with underscores
cat_encoded_features.columns = cat_encoded_features.columns.str.lower().str.replace(' ', '_')

In [74]:
# Reset index on encoded categorical features so rows align correctly when merging

cat_encoded_features = cat_encoded_features.reset_index(drop=True)
num_cleaned_features = df[num_cols_all].reset_index(drop=True)
target_feature = df['loan_status'].reset_index(drop=True)

In [100]:
# Combine loan_id, encoded categorical features, numerical features, and target feature

df_final = pd.concat([
    df['loan_id'].reset_index(drop=True),
    cat_encoded_features,
    num_cleaned_features,
    target_feature
], axis=1)

df_final

,loan_id,gender_male,married_yes,dependents_1.0,dependents_2.0,dependents_3.0,self_employed_yes,credit_history_1.0,loan_amount,loan_amount_term,applicant_income,coapplicant_income,loan_status
0,LP001002,1,0,0,0,0,0,1,128.0,360.0,5849,0.0,1
1,LP001003,1,1,1,0,0,0,1,128.0,360.0,4583,1508.0,0
2,LP001005,1,1,0,0,0,1,1,66.0,360.0,3000,0.0,1
3,LP001006,1,1,0,0,0,0,1,120.0,360.0,2583,2358.0,1
4,LP001008,1,0,0,0,0,0,1,141.0,360.0,6000,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
609,LP002978,0,0,0,0,0,0,1,71.0,360.0,2900,0.0,1
610,LP002979,1,1,0,0,1,0,1,40.0,180.0,4106,0.0,1
611,LP002983,1,1,1,0,0,0,1,253.0,360.0,8072,240.0,1
612,LP002984,1,1,0,1,0,0,1,187.0,360.0,7583,0.0,1


<h2>Feature Engineering</h2>

In [ ]:
#combine applicant and coapplicant income into a single feature
df_final['total_income'] = df_final['applicant_income'] + df_final['coapplicant_income']

#Check the first few rows of the new column
print(df_final[["applicant_income","coapplicant_income","total_income"]].head())

In [ ]:
# Changing the data type of 'loan_amount_term' to int64 and scaling 'loan_amount' by 1000
df_final['loan_amount_term'] = df_final['loan_amount_term'].astype('int64')
df_final['loan_amount'] = df_final['loan_amount'] * 1000
df_final[['loan_amount_term', 'loan_amount']].head()

In [ ]:
# Calculate the EMI
df_final['EMI'] = (df_final['loan_amount']/ df_final['loan_amount_term']).round(1)
print(df_final[['EMI']].head())

In [ ]:
# Calculate the balance income
df_final['balance_income'] = df_final['total_income'] - df_final['EMI']

# Print the first few rows of the updated columns
print(df_final[['total_income', 'EMI', 'balance_income']].head())

In [ ]:
# final Save after all feature engineering
df_final.to_csv("c:/Users/sello/OneDrive/Desktop/loan-predictor/notebooks.csv", index=False)

# Load the saved CSV
df_check = pd.read_csv("c:/Users/sello/OneDrive/Desktop/loan-predictor/notebooks.csv")

# Show the first 5 rows
print(df_check.head())